In [1]:
import numpy as np
from scipy.linalg import toeplitz,dft
import matplotlib.pyplot as plt

In [80]:
K = 2 ** np.random.randint(1, 4)
print(f'Number of message bits to be transmitted: {K}')
L = K
print(f'Channel length for multipath fading: {L}')

# channel variance
p = np.random.uniform(0,1)
print(f'Complex channel variance: {p}')
h_com = np.sqrt(p/2)*np.random.randn(1) + 1j*np.sqrt(p/2)*np.random.randn(1) 
print(f'Flat fading channel coefficient: {h_com}')
h_mag = np.abs(h_com)
h_phase = np.angle(h_com)/(2*np.pi)
print(f'Attenuation: {np.round(h_mag, 6)}, rotation: {np.round(h_phase, 6)}(in degrees) applied by channel')

h_real = p * np.random.randn(1)
print(f'Coefficient of slow fading channel: {h_real}')

Number of message bits to be transmitted: 8
Channel length for multipath fading: 8
Complex channel variance: 0.38342556513845827
Flat fading channel coefficient: [-0.81562083+0.10012143j]
Attenuation: [0.821743], rotation: [0.48056](in degrees) applied by channel
Coefficient of slow fading channel: [-0.2442985]


In [81]:
ita = np.random.uniform(0,0.5)
print(f'Autocorrelation sidelobe: {ita}')
discriminant = np.sqrt(1-4*ita**2)

# Ri = (1 - discriminant) / (2*ita)
# Ro = (1 + discriminant) / (2*ita)

R = np.sqrt(1 + np.sin(np.pi / K))
Ri, Ro = R**(-1), R

print(f'Radius of inner circle: {Ri}')
print(f'Radius of outer circle: {Ro}')

theta_k = (2 * np.pi)/K                                     #  + np.pi/4
print(f'Fundamental phase of sectors: {theta_k}')

Autocorrelation sidelobe: 0.14055664250994332
Radius of inner circle: 0.8504300947672564
Radius of outer circle: 1.1758756024193588
Fundamental phase of sectors: 0.7853981633974483


In [82]:
message = [np.random.randint(2) for i in range(K)]
print(f'The message to be transmitted: {message}')

# Codebook has all possible Zeros combinations
zero_geometry = [( Ri*np.exp(1j * theta_k * i), Ro*np.exp(1j * theta_k * i)) for i in range(K)]

print(f"\nGenerated codebook using huffman autocorrelation: {np.round(zero_geometry, 6)}")

tx_zeros = [zero_geometry[mk][message[mk]] for mk in range(K)]
print(f'\nZeroes selected wrt to message to be transmitted: {np.round(tx_zeros, 6)}')

The message to be transmitted: [1, 0, 1, 0, 1, 1, 0, 0]

Generated codebook using huffman autocorrelation: [[ 0.85043 +0.j        1.175876+0.j      ]
 [ 0.601345+0.601345j  0.83147 +0.83147j ]
 [ 0.      +0.85043j   0.      +1.175876j]
 [-0.601345+0.601345j -0.83147 +0.83147j ]
 [-0.85043 +0.j       -1.175876+0.j      ]
 [-0.601345-0.601345j -0.83147 -0.83147j ]
 [-0.      -0.85043j  -0.      -1.175876j]
 [ 0.601345-0.601345j  0.83147 -0.83147j ]]

Zeroes selected wrt to message to be transmitted: [ 1.175876+0.j        0.601345+0.601345j  0.      +1.175876j
 -0.601345+0.601345j -1.175876+0.j       -0.83147 -0.83147j
 -0.      -0.85043j   0.601345-0.601345j]


In [83]:
def toeplitz_iterator(zeros):

    for k in range(len(zeros)):
        if k == 0:
            c = np.array([[1, -zeros[k]]]).T   # (z-alpha)
        else:
            column = np.zeros(k+2, dtype=complex)
            column[0] = 1
            column[1] = -zeros[k]

            row = np.zeros(k+1, dtype=complex)
            row[0] = 1

            T = toeplitz(column, row)

            c = T @ c

    return c.flatten()

In [84]:
x = toeplitz_iterator(tx_zeros)
x = x[::-1]
print(f"The transmitted sequence will be: {x}")

#  --- to verify the methof is actually performing the coefficient generation
# x_z = np.roots(x)
# print(f'\n [DEBUG]Zeros Transmitted: {x_z}')

The transmitted sequence will be: [-1.        +9.43689571e-16j  0.23012473+9.53207823e-02j
 -0.35166171+3.07790374e-01j -0.12995573-6.17253348e-01j
 -0.70884047+2.63633176e-01j -0.34457145+5.28356607e-01j
 -0.30779037-3.51661712e-01j  0.23012473-9.53207823e-02j
  1.        +0.00000000e+00j]


In [85]:
# y = x * h_real
y = x * h_real
print(f'Received sequence through channel without noise: {y}')

#  --- to verify the methof is actually performing the coefficient generation
# y_zeros = np.roots(y)
# print(f'\n Zeros received: {y_zeros}')


Received sequence through channel without noise: [ 0.2442985 -2.30541950e-16j -0.05621913-2.32867245e-02j
  0.08591043-7.51927278e-02j  0.03174799+1.50794069e-01j
  0.17316867-6.44051904e-02j  0.08417829-1.29076728e-01j
  0.07519273+8.59104300e-02j -0.05621913+2.32867245e-02j
 -0.2442985 +0.00000000e+00j]


In [86]:
# FFT based MOCZ decoder - DiZeT Decoder
# Construction of Vandermode matrix (ML based decoder) - will not be done since we will be using FFT based

N_r = len(y)
print(f'Length of received Sequence: {N_r}')

N_tx_seq = len(zero_geometry)

Q = int( 2 ** np.ceil( np.log( np.ceil(N_r / N_tx_seq) ) / np.log(2) ) )
# Q = K
print(f'The oversampling factor: {Q}')

# modify the above command such that QK is power of 2

# zero geometry is known at RX

Length of received Sequence: 9
The oversampling factor: 2


In [87]:
N_fft = Q * K

DRo_diag = np.diag( Ro ** np.arange(N_r) )

col_vec = np.full( (DRo_diag.shape[0], 1), 0)
# print(col_vec)
N_append = N_fft - N_r
print(f'Number of zero column vector to be appended: {N_append}')

if N_append > 0:
    append_block = np.tile(col_vec, (1, int(N_append)))
    # print(append_block)
    DRo_diag_fft = np.hstack((DRo_diag, append_block))
else:
    DRo_diag_fft = DRo_diag

# print(DRo_diag_fft)

Number of zero column vector to be appended: 7


In [88]:
print(f'Size of FFT: {N_fft}')
fft = np.conjugate( dft(N_fft) )
print(f'The received sequence: \n{np.round(y, 6)}')
# y_conj_timeRev = [y[len(y)-1 - i] for i in range(len(y))]   # missed the conjugate of the sequence
y_conj_timeRev = np.conjugate(y[::-1])
print(f'The time-reversal conjugate of received sequence: \n{np.round(y_conj_timeRev, 6)}')

Size of FFT: 16
The received sequence: 
[ 0.244299-0.j       -0.056219-0.023287j  0.08591 -0.075193j
  0.031748+0.150794j  0.173169-0.064405j  0.084178-0.129077j
  0.075193+0.08591j  -0.056219+0.023287j -0.244299+0.j      ]
The time-reversal conjugate of received sequence: 
[-0.244299-0.j       -0.056219-0.023287j  0.075193-0.08591j
  0.084178+0.129077j  0.173169+0.064405j  0.031748-0.150794j
  0.08591 +0.075193j -0.056219+0.023287j  0.244299+0.j      ]


In [89]:
YRo_QK = y @ DRo_diag_fft @ fft
YRi_QK = y_conj_timeRev @ DRo_diag_fft @ fft

YRo = np.abs( YRo_QK[::Q] )
# YRi = ( Ro ** (K-1) ) * np.abs( YRi_QK[::Q] )
YRi = np.abs( YRi_QK[::Q])

# print(YRo)
# print(YRi)

# message_Rx = ( 1 + np.sign(YRo - YRi) ) / 2
message_Rx = (YRo < YRi).astype(int)
print(message_Rx)

[1 0 1 0 1 1 0 0]


In [90]:
# ---------   efficient way to implement this ----------

# a = [1, 2, 3, 4, 5]
# x = np.pad(a, (2,1), mode='constant')
# print(x)

scaling_vec = Ro ** np.arange(N_r)

y_scaled = y * scaling_vec

y_conj = np.conjugate(y[::-1])
y_conj_scaled = y_conj * scaling_vec

y_padded = np.pad(y_scaled, (0, N_fft - N_r), mode='constant')
Y_eval = np.abs( np.fft.ifft(y_padded) )

y_conj_padded = np.pad(y_conj_scaled, (0, N_fft - N_r), mode='constant')
Y_conj_eval = np.abs( np.fft.ifft(y_conj_padded) )

message_received = ( 1 - np.sign( Y_eval[::Q] - ( Y_conj_eval[::Q]) ) ) / 2    #  Ro**(K-1) *
print(message_received)

[1. 0. 1. 0. 1. 1. 0. 0.]


In [91]:
ber = np.mean(message != message_Rx)
print(f'BER: {ber}')
print(message)

BER: 0.0
[1, 0, 1, 0, 1, 1, 0, 0]


In [92]:
# m_rx_de = np.array([])

# def polynomial(x, alpha):
#     s = 1
#     for i in range(len(x)):
#         s *= x[i] * alpha**i
#     return np.abs(s)

# for k in range(len(zero_geometry)):
#     alpha_k = zero_geometry[k]
#     # print(alpha_k)
#     ak_0 = polynomial(y, alpha_k[0])
#     ak_1 = polynomial(y, alpha_k[1])
#     # print(ak_0)
#     if ak_0 < ak_1:
#         print(True)
#         m_rx_de += [0]
#     else:
#         np.append(m_rx_de, [1])

# print(m_rx_de)

In [93]:
# a = np.array([1, 2,3,4]).T
# F = dft(len(a))
# A = F @ a
# print(A)
# b = a[::-1]
# B = F @ b
# print(B)

# Fractional Frequency Offset Estimation using FFT

In [94]:
min_q = {}
for q in range(Q):
    sumZeros = 0
    for k in range(K):
        idx = (Q * k + q) % N_fft
        sumZeros += min( np.abs(YRi_QK[idx]), np.abs(YRo_QK[idx]) )
    min_q[q] = sumZeros
print(min_q)
q_est = np.argmin(min_q)
print(f'Estimatied sub-sector: {q_est}')
theta_est = ( q_est / Q ) * theta_k
print(f'Estimated fractional frequency offset: {theta_est}') 

{0: np.float64(9.780681915033831e-15), 1: np.float64(9.878820866055367)}
Estimatied sub-sector: 0
Estimated fractional frequency offset: 0.0


In [95]:
def fftCon(y, K, R):
    N_r = len(y)
    Q = int( 2 **( np.log( np.ceil(N_r/K) )/np.log(2) ) )
    N_fft = Q * K

    scaling_vec = R ** np.arange(N_r)
    y_ctr = np.conjugate(y[::-1])

    y_scaled = y * scaling_vec
    y_ctr_scaled = y_ctr * scaling_vec

    y_pad = np.pad(y_scaled, (0, N_fft - N_r), mode='constant')
    y_ctr_pad = np.pad(y_conj_scaled, (0, N_fft - N_r), mode='constant')

    Y_eval = np.abs( np.fft.ifft(y_pad) )
    Y_ctr_eval = np.abs( np.fft.ifft(y_ctr_pad) )

    return Y_eval, Y_ctr_eval

In [96]:
def ffo_est(y, Q, K, R):

    Yo, Yi = fftCon(y, K, R)
    
    min_q = {}
    for q in range(Q):
        sumZeros = 0
        for k in range(K):
            idx = (Q * k + q) % len(Yo)
            sumZeros += min( np.abs(Yi[idx]), np.abs(Yo[idx]) )
        min_q[q] = sumZeros
    q_est= np.argmin(min_q)
    print(f'Estimatied sub-sector: {q_est}')
    theta_est = ( q_est / Q ) * theta_k
    print(f'Estimated fractional frequency offset: {theta_est}') 
    return theta_est

In [97]:
def fftDizet(y, K, R, Q):

    Y_eval, Y_ctr_eval = fftCon(y, K, R)

    message_received = ( 1 - np.sign( Y_eval[::Q] - Y_ctr_eval[::Q] ) ) / 2 

    return message_received

In [ ]:
y_com_ch = x * h_com
print(f'Received sequence through complex slow fading channel without noise: {y_com_ch}')

theta_est = ffo_est(y_com_ch, Q, K, R)

# fractional offset correction
M_theta = np.diag( np.exp(-1j*theta_est) ** np.arange(N_r) )
# print(M_theta)
y_ffo = y_com_ch @ M_theta
# print(y_ffo)

#  Ro = np.sqrt(1 + np.sin(np.pi/K))
msg_Rx_ffo = fftDizet(y_ffo, len(zero_geometry), Ro, Q)     
msg_Rx_ffo = np.array(msg_Rx_ffo, dtype=int)
# print(msg_Rx_ffo)

ber = np.mean(message != msg_Rx_ffo)
print(f'BER: {ber}')
print(message)

Received sequence through complex slow fading channel without noise: [ 0.81562083-0.10012143j -0.19723817-0.0547052j   0.2560062 -0.28624911j
  0.16779489+0.49043333j  0.55174972-0.28599483j  0.22813983-0.46543764j
  0.28624911+0.2560062j  -0.17815087+0.10078603j -0.81562083+0.10012143j]
Estimatied sub-sector: 0
Estimated fractional frequency offset: 0.0
BER: 0.0
[1, 0, 1, 0, 1, 1, 0, 0]


In [99]:
def coeffCon(msg):
    K = len(msg)
    R = np.sqrt( 1 + np.sin(np.pi/K) )
    theta_k = (np.pi * 2)/K

    zeros = [zero_geometry[mk][msg[mk]] for mk in range(K)]
    
    x = toeplitz_iterator(zeros)     # x = [ x0, x1, x2, ....., xK]
    
    return x[::-1]

In [102]:
coeff_recon = coeffCon(msg_Rx_ffo)

h_est = y_com_ch / coeff_recon

print(f'Estimated channel coefficients: {h_est}')

print(f'\nComplex slow fading channel used: {h_com}')


Estimated channel coefficients: [-0.81562083+0.10012143j -0.81562083+0.10012143j -0.81562083+0.10012143j
 -0.81562083+0.10012143j -0.81562083+0.10012143j -0.81562083+0.10012143j
 -0.81562083+0.10012143j -0.81562083+0.10012143j -0.81562083+0.10012143j]

Complex slow fading channel used: [-0.81562083+0.10012143j]


In [ ]:
# inference to calculate PAPR

a = np.ones(10)
b = [1, -1] * 5
print(f"Sequence a is {a}")
a_fft = np.fft.fft(a)
print(a_fft)
print(f"Sequence b is {b}")
b_fft = np.fft.fft(b)
print(b_fft)

Sequence a is [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[10.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j
  0.+0.j]
Sequence b is [1, -1, 1, -1, 1, -1, 1, -1, 1, -1]
[ 0.00000000e+00+0.j -2.22044605e-16+0.j  0.00000000e+00+0.j
 -1.11022302e-16+0.j  0.00000000e+00+0.j  1.00000000e+01+0.j
  0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
 -1.11022302e-16+0.j]


In [19]:
a = np.random.rand(10)
print(a)
mini = np.argmin(a)
print(mini)
print(np.pad(a, (0,2), mode='constant'))

[0.25331075 0.07579574 0.53032867 0.02311559 0.65390921 0.06897249
 0.00452478 0.45150647 0.16591589 0.46639446]
6
[0.25331075 0.07579574 0.53032867 0.02311559 0.65390921 0.06897249
 0.00452478 0.45150647 0.16591589 0.46639446 0.         0.        ]


In [14]:
a = {0: 0.18, 1: 0.12, 2:0.13, 3:0.16, 4:0.17, 5:0.19}
print(min(a, key=a.get))
print(min(a.values()))

1
0.12


In [6]:
print(np.flip(np.arange(9)))

[8 7 6 5 4 3 2 1 0]
